<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://arize.com/docs/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://arize-ai.slack.com/join/shared_invite/zt-2w57bhem8-hq24MB6u7yE_ZF_ilOYSBw#/shared-invite/email">社区</a>
    </p>
</center>
<h1 align="center">使用基准数据集创建自定义 LLM 评估器</h1>

好的评估器应衡量你真正关心的内容；要确认这一点，必须使用你亲自判断过的示例进行测试。本 Notebook 按照内置评估器的构建方式，分三步创建一个以 LLM 为裁判的自定义评估器：

1. **构建基准数据集**：通过人工标注的示例明确你对“好”的定义。
2. **编写裁判提示词**：使用 LLM 裁判模板表达这一定义。
3. **验证后再信任**：衡量裁判与人工标签的一致程度，持续迭代直到达到要求。

完整说明见[实践指南](https://arize.com/docs/phoenix/cookbook/evaluation/creating-a-custom-llm-evaluator-with-a-benchmark-dataset)。

## 环境准备

安装依赖、连接 Phoenix，并自动插桩 OpenAI，将每次调用追踪到 `receipt-classification` 项目。

In [ ]:
%pip install -q arize-phoenix-client arize-phoenix-evals arize-phoenix-otel openinference-instrumentation-openai openai pandas

In [ ]:
import os
from getpass import getpass

from phoenix.otel import register

# 仅提示输入环境中尚未设置的密钥，避免覆盖已有值。
if not os.environ.get("PHOENIX_API_KEY"):
    os.environ["PHOENIX_API_KEY"] = getpass("请输入 Phoenix API 密钥：")
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("请输入 OpenAI API 密钥：")
# 使用 Phoenix Cloud 时，还需指定空间的收集器端点，例如：
# os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = "https://app.phoenix.arize.com/s/your-space"

tracer_provider = register(
    project_name="receipt-classification",
    auto_instrument=True,  # 为已安装的库插桩，包括 OpenAI
)

## 生成收据提取追踪

我们让 GPT-5.5 将收据图片转换为简短的报销报告。调用会自动追踪到 Phoenix，通常你会在其中进行标注。示例图片来自 [Wikimedia Commons](https://commons.wikimedia.org/wiki/Category:Receipts) 上的公开收据。

In [ ]:
from openai import OpenAI

client = OpenAI()


def extract_receipt_data(image_url):
    response = client.chat.completions.create(
        model="gpt-5.5",
        max_tokens=500,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": "分析这张收据，返回用于报销报告的简短摘要。仅包含费用类别、总金额和商品概要",
                    },
                    {"type": "image_url", "image_url": {"url": image_url}},
                ],
            }
        ],
    )
    return response.choices[0].message.content


urls = [
    "https://upload.wikimedia.org/wikipedia/commons/2/25/Receipt.jpg",
    "https://upload.wikimedia.org/wikipedia/commons/d/df/Save_Mart_recipt_2010-10-23.jpg",
    "https://upload.wikimedia.org/wikipedia/commons/9/9e/Restaurant_Bill_1_2013-07-08.jpg",
]
for url in urls:
    print(extract_receipt_data(url)[:120], "...\n")

## 构建基准数据集

实际使用时，你会在 Phoenix UI 中标注上面的追踪并将其导入数据集（参见[实践指南](https://arize.com/docs/phoenix/cookbook/evaluation/creating-a-custom-llm-evaluator-with-a-benchmark-dataset)）。为使本 Notebook 能完整运行，这里使用一个小型的**预标注**数据集。

注意：`create_dataset` 将 DataFrame 序列化为 CSV，因此 `attributes.llm.output_messages` 等嵌套列会存储为字符串。我们提前将模型输出展开为纯文本列，供评估器读取。

In [ ]:
import pandas as pd
from phoenix.client import Client

# 代替已标注追踪的数据：(输入、模型报销报告输出、人工标签)。
rows = [
    (
        "receipt_office.jpg",
        "Category: Office Supplies\nTotal: $47.32\nItems: paper, stapler, pens",
        "accurate",
    ),
    (
        "receipt_dinner.jpg",
        "Category: Meals\nTotal: $86.10\nItems: 2 entrees, 1 appetizer, 2 drinks",
        "accurate",
    ),
    ("receipt_vague.jpg", "Category: Misc\nTotal: $15.00\nItems: stuff", "almost accurate"),
    ("receipt_nototal.jpg", "Category: Travel\nSummary: round-trip train ticket", "inaccurate"),
    ("receipt_offtask.jpg", "The image shows a sunset over the ocean.", "inaccurate"),
]
full_df = pd.DataFrame(
    {
        "attributes.input.value": [r[0] for r in rows],
        "attributes.llm.output_messages": [[{"message.content": r[1]}] for r in rows],
        "result.label": [r[2] for r in rows],
    }
)


def first_message_content(messages):
    # get_spans_dataframe 以消息字典列表返回 output_messages；
    # 处理空行，以及偶尔已转换为字符串的值。
    if isinstance(messages, list) and messages:
        return messages[0].get("message.content", "")
    if isinstance(messages, str):
        return messages
    return ""


full_df["output_text"] = full_df["attributes.llm.output_messages"].apply(first_message_content)

px_client = Client()
dataset = px_client.datasets.create_dataset(
    name="annotated-receipts",
    dataframe=full_df,
    input_keys=["attributes.input.value"],
    output_keys=["output_text"],
    metadata_keys=["result.label"],
)

## 编写裁判提示词并运行实验

任务对每个示例运行裁判，评估器为它与人工标签的一致性打分。先使用一个宽松的模板建立基线。

In [ ]:
from phoenix.client.experiments import run_experiment
from phoenix.evals import LLM, ClassificationEvaluator

choices = ["accurate", "almost accurate", "inaccurate"]
llm = LLM(provider="openai", model="gpt-5.5")

initial_template = """你是评估器，负责评判模型生成的报销报告质量。
模型输出（报销报告）： {output}
选择以下一个标签，只输出标签：
- "accurate" - 包含费用类别、总金额和商品概要；所有信息看起来合理
- "almost accurate" - 基本正确但有小问题，例如缺少一个要素或类别模糊
- "inaccurate" - 存在重大错误或缺少关键信息
"""


def evaluate_response(output, metadata):
    return 1 if metadata["result.label"] == output else 0


def make_task(template):
    evaluator = ClassificationEvaluator(
        name="receipt_accuracy", prompt_template=template, llm=llm, choices=choices
    )

    def task_function(input, reference):
        output = reference["output_text"]
        return evaluator.evaluate(eval_input={"output": output})[0].label

    return task_function


initial_experiment = run_experiment(
    dataset=dataset,
    task=make_task(initial_template),
    evaluators=[evaluate_response],
    experiment_name="初始模板",
)

## 迭代裁判提示词

收紧规则，补上第一次运行暴露的缺口，然后重新运行以比较一致性。

In [ ]:
improved_template = """你是评估器，负责评判模型生成的报销报告质量。
模型输出（报销报告）： {output}
选择以下一个标签，只输出标签：
- "accurate" - 总价、明细列表和费用类别全部存在且合理，三项缺一不可。
- "almost accurate" - 基本正确但有小问题，例如类别过于模糊或缺少一个要素。
- "inaccurate" - 存在重大错误或缺少关键信息，例如完全缺少总价。
"""

improved_experiment = run_experiment(
    dataset=dataset,
    task=make_task(improved_template),
    evaluators=[evaluate_response],
    experiment_name="改进模板",
)

## 验证后再信任

每次实验都会报告裁判标签与人工标签匹配的频率，这个一致性分数就是验证信号。在 Phoenix 中比较两次运行：改进的模板应更频繁地与基准标签一致。只有与基准对齐后，才能在生产中信任裁判；结果不理想时，检查分歧并收紧提示词或完善人工标签。